# Otimização e simulação de cenários no QAI Control
**Pesquisa Operacional e Cálculo Aplicado**  
**Integrantes:** Gustavo Carvalho, Julia Egute, Pedro Perroni e Pablo Perez  

Este notebook reproduz os quatro cenários do relatório. Buscamos distribuir as horas de dois agentes de IA para maximizar o retorno semanal, respeitando orçamento, capacidade e um mínimo de utilização do Agente Avançado.



## 1 Preparação do ambiente


In [1]:
import sys
import subprocess
import tempfile
from pathlib import Path

# Carrega uma cópia isolada da versão compatível, mesmo se outra
# versão do PuLP já tiver sido importada nesta sessão.
pasta_pulp = Path(tempfile.gettempdir()) / "qai_pulp_330"
if not (pasta_pulp / "pulp" / "__init__.py").exists():
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--target", str(pasta_pulp), "pulp==3.3.0"
    ])
for nome_modulo in list(sys.modules):
    if nome_modulo == "pulp" or nome_modulo.startswith("pulp."):
        del sys.modules[nome_modulo]
sys.path.insert(0, str(pasta_pulp))

import pulp
import pandas as pd
from IPython.display import display
assert pulp.__version__ == "3.3.0"
print("PuLP:", pulp.__version__)
print("Solucionadores disponíveis:", pulp.listSolvers(onlyAvailable=True))

PuLP: 3.3.0
Solucionadores disponíveis: ['PULP_CBC_CMD', 'HiGHS']


## 2 Desenvolvimento do modelo de otimização
- $x_1$: horas semanais do Agente IA Básico.
- $x_2$: horas semanais do Agente IA Avançado.

As variáveis são contínuas e não negativas. No cenário base:

$$\max Z = 30x_1 + 50x_2$$

Sujeito a:

$$2x_1 + 5x_2 \leq 240$$
$$x_1 + x_2 \leq 60$$
$$x_2 \geq 10, \quad x_1,x_2 \geq 0$$

O custo por hora é de R$ 2 para o Básico e R$ 5 para o Avançado. Z representa retorno, não necessariamente lucro líquido: os custos limitam o orçamento, mas não são subtraídos da função objetivo.

In [2]:
def resolver_cenario(nome, orcamento, capacidade, retorno_basico, retorno_avancado):
    modelo = pulp.LpProblem("QAI_Control", pulp.LpMaximize)
    x1 = pulp.LpVariable("Horas_Basico", lowBound=0)
    x2 = pulp.LpVariable("Horas_Avancado", lowBound=0)

    modelo += retorno_basico*x1 + retorno_avancado*x2, "Retorno"
    modelo += 2*x1 + 5*x2 <= orcamento, "Orcamento"
    modelo += x1 + x2 <= capacidade, "Capacidade"
    modelo += x2 >= 10, "Governanca"
    modelo.solve(pulp.PULP_CBC_CMD(msg=False))

    status = pulp.LpStatus[modelo.status]
    if status != "Optimal":
        raise ValueError(f"{nome}: {status}")
    a, b = x1.varValue, x2.varValue
    custo = 2*a + 5*b
    folga_b = orcamento - custo
    folga_c = capacidade - a - b
    excesso = b - 10
    # Tolerância para pequenas diferenças numéricas do solucionador.
    tolerancia = 1e-4
    assert a >= -tolerancia and b >= 10 - tolerancia
    assert folga_b >= -tolerancia and folga_c >= -tolerancia
    ativas = []
    if abs(folga_b) < tolerancia:
        ativas.append("Orçamento")
    if abs(folga_c) < tolerancia:
        ativas.append("Capacidade")
    if abs(excesso) < tolerancia:
        ativas.append("Governança")
    return {
        "Cenário": nome, "Status": status,
        "Básico (h)": a, "Avançado (h)": b,
        "Retorno (R$)": pulp.value(modelo.objective),
        "Custo usado (R$)": custo, "Horas usadas": a+b,
        "Folga orçamento (R$)": folga_b,
        "Folga capacidade (h)": folga_c,
        "Excesso governança (h)": excesso,
        "Restrições ativas": " e ".join(ativas)
    }

## 3 Solução do cenário base
A conferência matemática fornece $x_1=20$ e $x_2=40$, com retorno de R$ 2.600,00. A solução dual da entrega anterior tem o mesmo valor, confirmando a otimalidade. Abaixo, resolvemos o problema com PuLP.

In [3]:
base = resolver_cenario("1 Base", 240, 60, 30, 50)
display(pd.DataFrame([base]).round(4))
assert abs(base["Retorno (R$)"] - 2600) < 1e-4
print("Conferência: 30 × 20 + 50 × 40 = R$ 2.600,00")

,Cenário,Status,Básico (h),Avançado (h),Retorno (R$),Custo usado (R$),Horas usadas,Folga orçamento (R$),Folga capacidade (h),Excesso governança (h),Restrições ativas
0,1 Base,Optimal,20.0,40.0,2600.0,240.0,60.0,0.0,0.0,30.0,Orçamento e Capacidade


Conferência: 30 × 20 + 50 × 40 = R$ 2.600,00


## 4 Simulação de cenários
Cada cenário parte do modelo base. As alterações não são acumuladas entre os cenários.

1. **Base:** orçamento de R$ 240, capacidade de 60 h e retornos de R$ 30 e R$ 50/h.
2. **Mais orçamento:** orçamento de R$ 270, mantendo os demais parâmetros.
3. **Queda do retorno:** retorno do Básico reduzido a R$ 15/h, com recursos da base.
4. **Expansão:** orçamento de R$ 300, capacidade de 80 h e retorno do Avançado de R$ 60/h.

Os custos por hora e o mínimo de 10 horas do Avançado permanecem iguais em todos os cenários.

In [4]:
cenarios = [
    ("1 Base", 240, 60, 30, 50),
    ("2 Mais orçamento", 270, 60, 30, 50),
    ("3 Queda do retorno", 240, 60, 15, 50),
    ("4 Expansão", 300, 80, 30, 60)
]
parametros = pd.DataFrame(cenarios, columns=[
    "Cenário", "Orçamento (R$)", "Capacidade (h)",
    "Retorno Básico (R$/h)", "Retorno Avançado (R$/h)"
])
display(parametros)
resultados = pd.DataFrame([resolver_cenario(*c) for c in cenarios])
resultados["Variação (R$)"] = resultados["Retorno (R$)"] - base["Retorno (R$)"]
resultados["Variação (%)"] = resultados["Variação (R$)"] / base["Retorno (R$)"] * 100

,Cenário,Orçamento (R$),Capacidade (h),Retorno Básico (R$/h),Retorno Avançado (R$/h)
0,1 Base,240,60,30,50
1,2 Mais orçamento,270,60,30,50
2,3 Queda do retorno,240,60,15,50
3,4 Expansão,300,80,30,60


## 5 Resultados obtidos nas simulações
Os cálculos preservam a precisão do solucionador. O arredondamento abaixo é apenas para apresentação.

In [5]:
display(resultados[["Cenário", "Status", "Básico (h)", "Avançado (h)",
                    "Retorno (R$)", "Variação (R$)", "Variação (%)",
                    "Restrições ativas"]].round(2))
display(resultados[["Cenário", "Custo usado (R$)", "Horas usadas",
                    "Folga orçamento (R$)", "Folga capacidade (h)",
                    "Excesso governança (h)"]].round(2))
# Verificação dos resultados apresentados no relatório.
esperados = [(20, 40, 2600), (10, 50, 2800), (0, 48, 2400),
             (100/3, 140/3, 3800)]
for (_, linha), (a, b, z) in zip(resultados.iterrows(), esperados):
    assert abs(linha["Básico (h)"] - a) < 1e-4
    assert abs(linha["Avançado (h)"] - b) < 1e-4
    assert abs(linha["Retorno (R$)"] - z) < 1e-3
print("Todos os cenários são ótimos, viáveis e consistentes com o relatório.")

,Cenário,Status,Básico (h),Avançado (h),Retorno (R$),Variação (R$),Variação (%),Restrições ativas
0,1 Base,Optimal,20.00,40.00,2600.0,0.0,0.00,Orçamento e Capacidade
1,2 Mais orçamento,Optimal,10.00,50.00,2800.0,200.0,7.69,Orçamento e Capacidade
2,3 Queda do retorno,Optimal,0.00,48.00,2400.0,-200.0,-7.69,Orçamento
3,4 Expansão,Optimal,33.33,46.67,3800.0,1200.0,46.15,Orçamento e Capacidade


,Cenário,Custo usado (R$),Horas usadas,Folga orçamento (R$),Folga capacidade (h),Excesso governança (h)
0,1 Base,240.0,60.0,0.0,0.0,30.00
1,2 Mais orçamento,270.0,60.0,0.0,0.0,40.00
2,3 Queda do retorno,240.0,48.0,0.0,12.0,38.00
3,4 Expansão,300.0,80.0,-0.0,0.0,36.67


Todos os cenários são ótimos, viáveis e consistentes com o relatório.


## 6 Análise gerencial
**Base.** As 20 horas do Básico e 40 do Avançado utilizam todo o orçamento e toda a capacidade. O retorno é de R$ 2.600,00.

**Mais orçamento.** O acréscimo de R$ 30 permite substituir 10 horas do Básico por 10 do Avançado. O retorno aumenta R$ 200, chegando a R$ 2.800. A variação corresponde ao preço-sombra do orçamento: $(20/3)\times30=200$. A aplicação é válida para essa alteração isolada, dentro do intervalo de sensibilidade de R$ 150 a R$ 300.

**Queda do retorno.** O Básico passa a gerar R$ 7,50 por real de orçamento, enquanto o Avançado gera R$ 10. O modelo aloca 48 horas ao Avançado e nenhuma ao Básico. O orçamento é totalmente utilizado e sobram 12 horas de capacidade. O retorno cai para R$ 2.400.

**Expansão.** A alocação de $100/3$ horas ao Básico e $140/3$ ao Avançado gera R$ 3.800, aumento de 46,15% sobre a base. Orçamento e capacidade são totalmente utilizados. Como três parâmetros mudaram, o ganho não pode ser atribuído somente à capacidade. Os preços-sombra da base não devem ser aplicados diretamente a essa nova configuração.

**Conclusão.** O cenário 4 tem o maior retorno entre os avaliados. Para decidir pela expansão, ainda é necessário avaliar os custos de implantação e confirmar os retornos por hora. O modelo apoia o planejamento dos recursos; não mede dependência de IA nem valida a suficiência da regra de governança.